In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

data = pd.read_csv("/content/synthetic_tickets_1000.csv")

X = data['text']
y = data['category']

vectorizer = TfidfVectorizer()
X_vec = vectorizer.fit_transform(X)

model = LogisticRegression()
model.fit(X_vec, y)

LogisticRegression()

In [ ]:
def predict_category(text):
    text_vec = vectorizer.transform([text])
    return model.predict(text_vec)[0]

In [ ]:
predict_category("the display panel is broken")

'Hardware'

In [ ]:
def detect_priority(text):
    high_words = ["urgent", "server down", "crash", "immediately"]

    for word in high_words:
        if word in text.lower():
            return "High"
    return "Medium"

In [ ]:
priority = [predict_category("the display panel is broken"), predict_category("The page is not loading"),predict_category("the page is hang")]

In [ ]:
for i in priority:
  print(detect_priority(i))

Medium
Medium
Medium


Dataset generated successfully: synthetic_tickets_1000.csv


In [ ]:
import pandas as pd
import pickle
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split


data = pd.read_csv("/content/synthetic_tickets_1000.csv")

X = data["text"]
y = data["category"]


domain_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("clf", LogisticRegression(max_iter=1000))
])

domain_pipeline.fit(X, y)

pickle.dump(domain_pipeline, open("domain_model.pkl", "wb"))

print("Domain model trained and saved.")



HIGH_KEYWORDS = [
    "urgent", "immediately", "asap",
    "server down", "system crash",
    "ransomware", "security breach",
    "not working", "critical", "downtime", "data loss", "critical failure"
]

MEDIUM_KEYWORDS = [
    "slow", "error", "issue",
    "problem", "warning", "glitch", "performance", "bug", "minor"
]

def detect_priority(text):
    text = text.lower()

    for word in HIGH_KEYWORDS:
        if word in text:
            return "High"

    for word in MEDIUM_KEYWORDS:
        if word in text:
            return "Medium"

    return "Low"




def analyze_ticket(content):
    # Load model
    model = pickle.load(open("domain_model.pkl", "rb"))

    domain = model.predict([content])[0]
    priority = detect_priority(content)

    result = {
        "content": content,
        "domain": domain,
        "priority": priority
    }

    return result




if __name__ == "__main__":
    sample = "Server down urgently need fix immediately"
    print(analyze_ticket(sample))

Domain model trained and saved.
{'content': 'Server down urgently need fix immediately', 'domain': 'Network', 'priority': 'High'}


In [ ]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
import torch
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset

# Load dataset
df = pd.read_csv("/content/synthetic_tickets_1000.csv")

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["category"])

dataset = Dataset.from_pandas(df)

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

dataset = dataset.map(tokenize, batched=True)

dataset = dataset.rename_column("label", "labels")
dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=6
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=16
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=126, training_loss=0.584421127561539, metrics={'train_runtime': 97.7679, 'train_samples_per_second': 20.457, 'train_steps_per_second': 1.289, 'total_flos': 264953696256000.0, 'train_loss': 0.584421127561539, 'epoch': 2.0})